<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z321_McNemar_ARIMA_vs_AutoGluon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# McNemar: AutoARIMA vs AutoGluon

## ¿Para qué sirve el Test de McNemar acá?

McNemar es un test de hipótesis para comparar dos clasificadores sobre los **mismos casos pareados**. Adaptado a este problema:

- Para cada uno de los 780 productos, definimos quién "ganó": el modelo cuya predicción estuvo **más cerca del valor real** (menor error absoluto)
- McNemar testa si la diferencia en la cantidad de productos ganados es **estadísticamente significativa** o podría ser azar

La tabla de contingencia es:

```
                     AutoGluon correcto   AutoGluon incorrecto
ARIMA correcto            n_11                   n_10
ARIMA incorrecto          n_01                   n_00
```

Solo importan los casos donde los modelos **discrepan** (`n_10` y `n_01`). Si `n_10 ≈ n_01` → diferencia no significativa. Si uno domina → hay diferencia real.

## Estructura del notebook

- **Parte 1**: Comparación exploratoria de predicciones (ya disponible con los CSVs)
- **Parte 2**: Test de McNemar completo (requiere valores reales de `202002` — se completa cuando estén disponibles)

## 0.1 Init

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import chi2

import warnings
warnings.filterwarnings('ignore')

Subir los dos archivos CSV de predicciones al entorno de Colab antes de correr esta celda.

In [ ]:
# cargar las predicciones de cada modelo
arima  = pl.read_csv('autoARIMA.csv').rename({'tn': 'tn_arima'})
gluon  = pl.read_csv('AutoGluon_RMSE.csv').rename({'tn': 'tn_gluon'})

# unir por product_id
tb = arima.join(gluon, on='product_id', how='inner')

print(f"Productos en comun: {tb.height}")
display(tb.head(10))

# Parte 1 — Comparacion exploratoria de predicciones

Antes de tener los valores reales podemos comparar **cuánto difieren las predicciones** entre sí. Una gran diferencia entre modelos en un producto indica que vale la pena investigar ese caso.

## 1.1 Diferencia absoluta y relativa por producto

In [ ]:
tb = tb.with_columns([
    (pl.col('tn_arima') - pl.col('tn_gluon')).alias('diff'),
    (pl.col('tn_arima') - pl.col('tn_gluon')).abs().alias('diff_abs'),
    # diferencia relativa respecto al promedio de las dos predicciones
    (
        (pl.col('tn_arima') - pl.col('tn_gluon')).abs() /
        ((pl.col('tn_arima') + pl.col('tn_gluon')) / 2 + 1e-9)
    ).alias('diff_rel')
])

print("Estadisticas de la diferencia (ARIMA - AutoGluon):")
display(tb.select(['diff', 'diff_abs', 'diff_rel']).describe())

## 1.2 Scatter: prediccion ARIMA vs AutoGluon

Si los dos modelos coinciden, los puntos caen sobre la diagonal. Los puntos alejados son los productos donde más discrepan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = tb['tn_arima'].to_numpy()
y = tb['tn_gluon'].to_numpy()

# escala normal
axes[0].scatter(x, y, alpha=0.4, s=15, color='steelblue')
lim = max(x.max(), y.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='línea perfecta')
axes[0].set_xlabel('AutoARIMA')
axes[0].set_ylabel('AutoGluon')
axes[0].set_title('Predicciones: ARIMA vs AutoGluon')
axes[0].legend()

# escala log para ver mejor los productos pequeños
axes[1].scatter(np.log1p(x), np.log1p(y), alpha=0.4, s=15, color='steelblue')
lim_log = max(np.log1p(x).max(), np.log1p(y).max()) * 1.05
axes[1].plot([0, lim_log], [0, lim_log], 'r--', linewidth=1, label='línea perfecta')
axes[1].set_xlabel('log(1 + AutoARIMA)')
axes[1].set_ylabel('log(1 + AutoGluon)')
axes[1].set_title('Predicciones en escala log')
axes[1].legend()

plt.tight_layout()
plt.show()

## 1.3 ¿Quién predice más alto?

Para cada producto: ¿ARIMA o AutoGluon predice más tn?

In [ ]:
n_arima_mayor = (tb['diff'] > 0).sum()
n_gluon_mayor = (tb['diff'] < 0).sum()
n_iguales     = (tb['diff'] == 0).sum()

print(f"ARIMA predice MAS que AutoGluon : {n_arima_mayor} productos ({100*n_arima_mayor/tb.height:.1f}%)")
print(f"AutoGluon predice MAS que ARIMA : {n_gluon_mayor} productos ({100*n_gluon_mayor/tb.height:.1f}%)")
print(f"Iguales                         : {n_iguales} productos")

fig, ax = plt.subplots(figsize=(10, 4))
diff_vals = tb['diff'].to_numpy()
ax.hist(diff_vals, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='sin diferencia')
ax.set_xlabel('ARIMA - AutoGluon  (tn predichas)')
ax.set_ylabel('cantidad de productos')
ax.set_title('Distribucion de diferencias entre predicciones')
ax.legend()
plt.tight_layout()
plt.show()

## 1.4 Top 10 productos con mayor discrepancia relativa

In [ ]:
top_discrepancia = tb.sort('diff_rel', descending=True).head(10)
display(
    top_discrepancia.select(['product_id', 'tn_arima', 'tn_gluon', 'diff_abs', 'diff_rel'])
)

# Parte 2 — Test de McNemar

**Esta sección requiere los valores reales de 202002.** Una vez que Kaggle publique el leaderboard final o tengas acceso al ground truth, completar la celda de carga de datos reales y correr el resto.

## ¿Cómo funciona McNemar?

Para cada producto definimos un ganador: el modelo con menor error absoluto respecto al valor real.

```
error_arima[i]  = |tn_real[i] - tn_arima[i]|
error_gluon[i]  = |tn_real[i] - tn_gluon[i]|

gana_arima[i]   = 1 si error_arima < error_gluon, 0 si no
gana_gluon[i]   = 1 si error_gluon < error_arima, 0 si no
```

Tabla de contingencia (solo los casos donde discrepan):
- `n_10`: ARIMA gana, AutoGluon pierde
- `n_01`: AutoGluon gana, ARIMA pierde

Estadístico: `χ² = (|n_10 - n_01| - 1)² / (n_10 + n_01)` con corrección de continuidad de Yates.

H₀: los dos modelos se equivocan en los mismos productos (no hay diferencia significativa).

In [ ]:
# ---------------------------------------------------------------
# COMPLETAR: cargar los valores reales de 202002
# Formato esperado: CSV con columnas product_id, tn
# ---------------------------------------------------------------
# tb_real = pl.read_csv('valores_reales_202002.csv').rename({'tn': 'tn_real'})
# ---------------------------------------------------------------

# descomenta las lineas de abajo cuando tengas el archivo
print("[PENDIENTE] Cargar tb_real con los valores reales de 202002 para correr el McNemar")

In [ ]:
# ---------------------------------------------------------------
# CORRER CUANDO tb_real ESTE DISPONIBLE
# ---------------------------------------------------------------

# tb_completo = tb.join(tb_real, on='product_id', how='inner')
#
# tb_completo = tb_completo.with_columns([
#     (pl.col('tn_real') - pl.col('tn_arima')).abs().alias('err_arima'),
#     (pl.col('tn_real') - pl.col('tn_gluon')).abs().alias('err_gluon'),
# ])
#
# tb_completo = tb_completo.with_columns([
#     (pl.col('err_arima') < pl.col('err_gluon')).alias('gana_arima'),
#     (pl.col('err_gluon') < pl.col('err_arima')).alias('gana_gluon'),
# ])
#
# n_10 = (tb_completo['gana_arima'] & ~tb_completo['gana_gluon']).sum()  # ARIMA gana, Gluon pierde
# n_01 = (tb_completo['gana_gluon'] & ~tb_completo['gana_arima']).sum()  # Gluon gana, ARIMA pierde
# n_11 = (tb_completo['gana_arima'] & tb_completo['gana_gluon']).sum()   # empate exacto
# n_00 = (~tb_completo['gana_arima'] & ~tb_completo['gana_gluon']).sum() # error igual
#
# print(f"Tabla de contingencia McNemar")
# print(f"  ARIMA gana, Gluon pierde (n_10): {n_10}")
# print(f"  Gluon gana, ARIMA pierde (n_01): {n_01}")
# print(f"  Ambos correctos (n_11)         : {n_11}")
# print(f"  Empate en error (n_00)          : {n_00}")
#
# # estadistico con correccion de Yates
# chi2_stat = (abs(n_10 - n_01) - 1)**2 / (n_10 + n_01)
# pvalue = 1 - chi2.cdf(chi2_stat, df=1)
#
# print(f"\nχ² = {chi2_stat:.4f}")
# print(f"p-value = {pvalue:.4f}")
# if pvalue < 0.05:
#     ganador = 'AutoARIMA' if n_10 > n_01 else 'AutoGluon'
#     print(f"Diferencia SIGNIFICATIVA (p < 0.05) → {ganador} es mejor")
# else:
#     print(f"Diferencia NO significativa (p >= 0.05) → no hay evidencia de que un modelo sea mejor")

print("[PENDIENTE] Descomentar el bloque cuando tb_real esté disponible")

## 2.1 Visualizacion de errores por producto (cuando tb_real esté disponible)

Cuando tengas los valores reales, esta celda grafica:
- Error por producto para cada modelo
- Quién ganó en cada producto (rojo = ARIMA, azul = AutoGluon)

In [ ]:
# ---------------------------------------------------------------
# CORRER CUANDO tb_completo ESTE DISPONIBLE
# ---------------------------------------------------------------

# tb_plot = tb_completo.sort('err_arima', descending=True)
#
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))
#
# # scatter errores: cada punto es un producto
# axes[0].scatter(
#     np.log1p(tb_plot['err_arima'].to_numpy()),
#     np.log1p(tb_plot['err_gluon'].to_numpy()),
#     alpha=0.4, s=15, color='steelblue'
# )
# lim = max(np.log1p(tb_plot['err_arima'].to_numpy()).max(),
#            np.log1p(tb_plot['err_gluon'].to_numpy()).max()) * 1.05
# axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='empate')
# axes[0].set_xlabel('log(1 + error ARIMA)')
# axes[0].set_ylabel('log(1 + error AutoGluon)')
# axes[0].set_title('Error por producto (escala log)\nArriba diagonal = AutoGluon gana')
# axes[0].legend()
#
# # barras: cuántos productos gana cada modelo por rango de ventas
# colores = ['tomato' if g else 'steelblue' for g in tb_plot['gana_arima'].to_list()]
# axes[1].bar(range(len(tb_plot)), tb_plot['err_arima'].to_numpy() - tb_plot['err_gluon'].to_numpy(),
#             color=colores, alpha=0.7, width=1.0)
# axes[1].axhline(0, color='black', linewidth=0.8)
# axes[1].set_xlabel('productos (ordenados por error ARIMA)')
# axes[1].set_ylabel('err_arima - err_gluon')
# axes[1].set_title('Diferencia de errores por producto\nRojo = ARIMA mejor, Azul = AutoGluon mejor')
#
# plt.tight_layout()
# plt.show()

print("[PENDIENTE] Descomentar el bloque cuando tb_completo esté disponible")